In [ ]:
!nvidia-smi

Sat Apr 26 15:08:03 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Check CUDA version
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


In [ ]:
# Install CUDA compiler
!apt update -qq
!apt install -y cuda


35 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  cpp-12 cuda-12-8 cuda-cccl-12-8 cuda-command-line-tools-12-8
  cuda-compiler-12-8 cuda-crt-12-8 cuda-cudart-12-8 cuda-cudart-dev-12-8
  cuda-cuobjdump-12-8 cuda-cupti-12-8 cuda-cupti-dev-12-8 cuda-cuxxfilt-12-8
  cuda-demo-suite-12-8 cuda-documentation-12-8 cuda-driver-dev-12-8
  cuda-gdb-12-8 cuda-libraries-12-8 cuda-libraries-dev-12-8 cuda-nsight-12-8
  cuda-nsight-compute-12-8 cuda-nsight-systems-12-8 cuda-nvcc-12-8
  cuda-nvdisasm-12-8 cuda-nvml-dev-12-8 cuda-nvprof-12-8 cuda-nvprune-12-8
  cuda-nvrtc-12-8 cuda-nvrtc-dev-12-8 cuda-nvtx-12-8 cuda-nvvm-12-8
  cuda-nvvp-12-8 cuda-o

In [ ]:
%%writefile chudnovsky.cu
#include <iostream>
#include <cuda_runtime.h>
#include <chrono>
#include <math.h>

#define N 10000  // Number of terms (can increase for better Pi approximation)

__device__ double factorial_device(int n) {
    double result = 1.0;
    for (int i = 2; i <= n; i++) {
        result *= i;
    }
    return result;
}

__global__ void chudnovsky_gpu(double *terms) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;
    if (idx < N) {
        double k = (double)idx;

        double numerator = factorial_device(6*k) * (545140134.0*k + 13591409.0);
        double denominator = factorial_device(3*k) * pow(factorial_device(k), 3) * pow(640320.0, 3*k);

        terms[idx] = numerator / denominator;
    }
}

double factorial_cpu(int n) {
    double result = 1.0;
    for (int i = 2; i <= n; i++) {
        result *= i;
    }
    return result;
}

void chudnovsky_cpu(double *terms) {
    for (int i = 0; i < N; i++) {
        double k = (double)i;

        double numerator = factorial_cpu(6*k) * (545140134.0*k + 13591409.0);
        double denominator = factorial_cpu(3*k) * pow(factorial_cpu(k), 3) * pow(640320.0, 3*k);

        terms[i] = numerator / denominator;
    }
}

int main() {
    double *h_terms_cpu = new double[N];
    double *h_terms_gpu = new double[N];
    double *d_terms;

    cudaMalloc(&d_terms, N * sizeof(double));

    // --- CPU computation ---
    auto start_cpu = std::chrono::high_resolution_clock::now();
    chudnovsky_cpu(h_terms_cpu);
    auto end_cpu = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> cpu_duration = end_cpu - start_cpu;

    // --- GPU computation ---
    auto start_gpu = std::chrono::high_resolution_clock::now();
    int threads_per_block = 256;
    int blocks_per_grid = (N + threads_per_block - 1) / threads_per_block;
    chudnovsky_gpu<<<blocks_per_grid, threads_per_block>>>(d_terms);
    cudaDeviceSynchronize();
    auto end_gpu = std::chrono::high_resolution_clock::now();

    cudaMemcpy(h_terms_gpu, d_terms, N * sizeof(double), cudaMemcpyDeviceToHost);
    std::chrono::duration<double, std::milli> gpu_duration = end_gpu - start_gpu;

    // --- Printing Outputs ---
    std::cout << "First 5 terms from CPU:" << std::endl;
    for (int i = 0; i < 5; i++) {
        std::cout << "h_terms_cpu[" << i << "] = " << h_terms_cpu[i] << std::endl;
    }

    std::cout << "\nFirst 5 terms from GPU:" << std::endl;
    for (int i = 0; i < 5; i++) {
        std::cout << "h_terms_gpu[" << i << "] = " << h_terms_gpu[i] << std::endl;
    }

    std::cout << "\nCPU Time: " << cpu_duration.count() << " ms" << std::endl;
    std::cout << "GPU Time: " << gpu_duration.count() << " ms" << std::endl;

    if (gpu_duration.count() > 0) {
        std::cout << "Speedup (CPU time / GPU time): " << cpu_duration.count() / gpu_duration.count() << "x" << std::endl;
    } else {
        std::cout << "GPU time too fast to measure accurately." << std::endl;
    }

    // --- Cleanup ---
    cudaFree(d_terms);
    delete[] h_terms_cpu;
    delete[] h_terms_gpu;

    return 0;
}


Overwriting chudnovsky.cu


In [ ]:
!nvcc -arch=sm_75 chudnovsky.cu -o chudnovsky

In [ ]:
!./chudnovsky

First 5 terms from CPU:
h_terms_cpu[0] = 1.35914e+07
h_terms_cpu[1] = 2.55384e-07
h_terms_cpu[2] = 1.33184e-21
h_terms_cpu[3] = 7.44345e-36
h_terms_cpu[4] = 4.3275e-50

First 5 terms from GPU:
h_terms_gpu[0] = 1.35914e+07
h_terms_gpu[1] = 2.55384e-07
h_terms_gpu[2] = 1.33184e-21
h_terms_gpu[3] = 7.44345e-36
h_terms_gpu[4] = 4.3275e-50

CPU Time: 1495.87 ms
GPU Time: 49.4246 ms
Speedup (CPU time / GPU time): 30.2657x
